# 1.3 TileLang与昇腾950

## 概述

上一节认识了TileLang的特点与使用场景。现在把目光转向950：谁来执行计算，数据放在哪里，又为什么要把数据分成块？

先认识硬件中的几个角色，再用一次加法理解tile，最后把TileLang放回整个软件体系中看。


## 1. 一次计算，需要哪些硬件配合

运行示例时，CPU上的Python程序负责准备数据、发起调用和检查结果，950负责执行设备侧计算。CPU这一侧称为<strong>Host（主机端）</strong>，950这一侧称为<strong>Device（设备端）</strong>。

先看图的上半部分：Host负责组织和发起任务，Device承担设备侧计算。再看Device内部：左侧是存放完整输入与输出的GM，右侧展开了本章加法所用的向量计算路径。

<img alt="Host与昇腾950的分工，以及GM、UB、向量寄存器和Vector之间的输入与结果流向" src="images/01.03_hardware_roles_v4.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

**计算由谁完成**

**Vector主要承担向量计算，Cube主要承担矩阵乘加。** 本章的加法使用Vector，后面学习矩阵乘法时再认识Cube。DMA搬运单元则负责相应的数据传输，把计算需要的数据送到所需位置。

**数据存放在哪里**

设备上的完整输入与输出通常位于<strong>GM（Global Memory，全局内存）</strong>。向量计算前，当前要处理的数据会先搬到片上的<strong>UB（Unified Buffer，统一缓冲区）</strong>，再加载到向量寄存器，由Vector执行运算。结果可以经UB写回GM。

沿图中的蓝色箭头看，输入从GM进入UB，再加载到向量寄存器中参与运算；沿绿色箭头看，结果从寄存器经UB写回GM。**UB和寄存器用于存放数据，Vector运算单元负责计算。** 把这些职责分清楚，后面看到搬运和计算代码时，就能知道每一步发生在哪里。

<p class="course-note"><small>本图按本章的SIMD示例简化绘制，省略了控制、同步与矩阵计算的数据通路。更完整的硬件关系可参考<a href="https://asc.gitcode.com/assets/simt_arch.BKF-UTiq.png">SIMD与SIMT混合编程硬件架构示意图</a>，来源为AscendC文档<a href="https://asc.gitcode.com/guide/programming_guide/advanced_programming/advanced_ai_core_programming_model/simd_simt_hybrid_programming/abstract_hardware_architecture.html">《抽象硬件架构》</a>中的图1。</small></p>


## 2. tile：组织计算的数据块

**在TileLang中，一块按一定形状组织的数据称为tile。** 它可以是一维数组中的一段，也可以是矩阵中的一个矩形区域。程序围绕这些数据块安排搬运和计算。

### 按数据块安排任务

为什么要分块？**片上存储容量有限，一次通常先处理完整输入中的一部分。** 对于相互独立的计算，还可以把不同部分安排给不同任务。

还是看加法。假设A、B各有1024个数，我们把它们各分成四块，安排四个任务，每个任务负责其中256个位置的加法。图中的橙色部分就是其中一份工作。

<img alt="将1024项加法划分为四份任务，每份处理对应的256项" src="images/01.03_tile_dataflow_v4.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

这个任务从A和B中各取一块，送到片上参与计算，再把结果写回C的对应位置。其他三个任务执行相同的过程，只是处理的数据范围不同。**tile描述的是一块数据，任务负责处理这块数据。**

### 块大小与性能

**tile的划分也会影响算子的性能。** 在本例中，每个任务处理一块数据。如果每块由256项增大到512项，任务数就从4个变为2个，但每个任务需要暂存的数据也更多。分块会影响片上存储占用、数据搬运和任务分配；矩阵计算还要考虑块的形状，以及数据能否被重复利用。

**块更大不一定更快，合适的划分需要结合算子、输入形状和硬件，通过实际测量来选择。** 本节先建立这一认识，后续向量与矩阵算子章节再结合实例讨论分块。

### 块内的向量计算

块内还可以用向量指令处理一组数。例如，把两组数据中对应位置的数相加，可以写成：

```python
c_vec = T.simd.vadd(a_vec, b_vec, mask)
```

这里的`a_vec`和`b_vec`各保存一组数据，`vadd`对它们执行向量加法。这种“一条指令处理多个数据”的方式称为**SIMD**，本课程采用显式的SIMD编程方式。

因此，数据块划定了这份任务要处理的范围，向量指令完成其中的具体计算。两者联系起来，就形成了本例的基本过程：搬入一块数据，用向量指令计算，再写回结果。

<details>
<summary>展开对照：这块数据怎样经过硬件</summary>

<img alt="当前数据块从GM经过UB和向量寄存器完成加法，再将结果写回GM" src="images/01.03_simd_dataflow_v3.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

沿着输入到输出的箭头，可以把前面认识的GM、UB、向量寄存器和Vector连接起来。图中向量操作每轮处理的是数据块中的一组数据。

</details>

<p class="course-note"><small>本例的四块划分用于演示，不是唯一分法，任务数也不等于物理核数。这里的代码只展示向量加法这一行；完整示例见1.4，任务索引、循环和掩码参数在后续章节展开。</small></p>


## 3. TileLang在软件体系中的位置

硬件的分工清楚后，再看示例中的几种软件分别做什么。

**准备输入和校验结果**

PyTorch与torch_npu让Python程序能够创建和使用NPU张量。示例用它们准备输入，也用PyTorch的计算结果检查自定义算子的输出。

**描述算子的数据与计算**

**TileLang处在算子编程这一层。** 我们用它描述设备要处理哪些数据块、怎样搬运、执行什么计算。它采用Python语法，但950执行的是编译后的设备程序。

**生成并运行设备程序**

这份描述会通过所选后端转换成设备代码。本课程首例采用AscendC后端；TileLang也可以通过相应后端生成PTO代码。相关编译工具和CANN运行支持共同帮助程序在950上执行。

<img alt="TileLang位于算子编程层，通过所选后端生成设备代码，由编译与运行工具支持在昇腾950上执行" src="images/01.03_software_layers_v4.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

图中橙色区域就是我们编写TileLang算子的位置。上方的Python程序负责准备输入、调用算子与检查结果，下方的工具负责把设备代码编译成程序，并支持它在950上执行。**学习TileLang，主要是在学习怎样描述这一层中的数据与计算安排。**

<p class="course-note"><small>蓝色实线表示设备程序的生成关系，绿色虚线表示调用已编译算子的运行时路径。</small></p>

到这里，既知道了TileLang负责描述什么，也知道了数据块怎样对应设备上的工作。下一节[1.4「环境准备与第一个算子」](01.04_first_operator.ipynb)，先运行一个完整的小例子；程序结构和接口规则在第2章继续学习。
